This notebook explains how to generate K-folds for cross-validation using `scikit-learn` for evaluation of machine learning models with out of sample data.

This notebook will work with an OpenML dataset to predict who pays for internet with 10108 observations and 69 columns.

### Packages

This tutorial uses:
* [pandas](https://pandas.pydata.org/docs/)
* [scikit-learn](https://scikit-learn.org/stable/)
    * [sklearn.datasets](https://scikit-learn.org/stable/datasets.html)
    * [sklearn.model_selection](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd
from sklearn.model_selection import KFold

## Reading the data

The data is from [OpenML](https://www.openml.org/d/981) imported using the Python package `sklearn.datasets`.

In [ ]:
data = fetch_openml(name='kdd_internet_usage', as_frame=True)
df = data.frame 
df.info()

In [ ]:
df.head()

In [ ]:
## check is there any missing values in this 
df.isnull().sum()

## Split the data into target and features.

Drop target leakage features of other options to pay.

In [ ]:
target = 'Who_Pays_for_Access_Work'
y = df[target]
X = data.data.drop(columns=['Who_Pays_for_Access_Dont_Know',
       'Who_Pays_for_Access_Other', 'Who_Pays_for_Access_Parents',
       'Who_Pays_for_Access_School', 'Who_Pays_for_Access_Self'])

In [ ]:
X

In [ ]:
y

## Cross-validation splitting

Scikit-learn's `KFold` will randomly sample the data into **N** folds (default of 5) that can be used to perform cross-validation during machine learning training.

In [ ]:
# Importing necessary libraries
from sklearn.model_selection import KFold

# Initializing KFold cross-validator with 10 splits, random state 1066, and shuffle set to True
kf = KFold(n_splits=10, random_state=1066, shuffle=True)

# Iterating through the splits generated by KFold
for train_index, test_index in kf.split(X):
    # Printing the indices of training and testing sets for each split
    print("Train:", train_index, "Test:", test_index)
    
    # Extracting the training features using iloc with the current train indices
    X_train = X.iloc[train_index, :]
    
    # Extracting the corresponding training labels using the current train indices
    y_train = y[train_index]
    
    # Extracting the test features using iloc with the current test indices
    X_test = X.iloc[test_index, :]
    
    # Extracting the corresponding test labels using the current test indices
    y_test = y[test_index]


## lets see how do modeling and cross validation part at same time

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder 
from sklearn.metrics import accuracy_score

# Lists to store the accuracy of each fold
fold_accuracies = []


# Initialize the model (for example, a RandomForestClassifier)
model = RandomForestClassifier()

# Initialize KFold cross-validator with 10 splits, random state 1066, and shuffle set to True
kf = KFold(n_splits=10, random_state=1066, shuffle=True)

# Initialize the label encoder
label_encoder = LabelEncoder()

# Identify categorical columns in your features (X) and apply label encoding
categorical_cols = X.select_dtypes(include=['category']).columns
for column in categorical_cols:
    X[column] = label_encoder.fit_transform(X[column])

# Lists to store the predictions and actual labels
predictions = []
true_labels = []

# Iterate through the splits generated by KFold
for train_index, test_index in kf.split(X):
    # Extract the training and testing data for this fold
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y[train_index], y[test_index]
    
    # Train the model on the training data
    model.fit(X_train, y_train)
    
    # Make predictions on the test data
    fold_predictions = model.predict(X_test)
    
    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_test, fold_predictions)
    
    # Append the accuracy to the fold_accuracies list
    fold_accuracies.append(accuracy)
    
    print(f'Accuracy: {accuracy:.2f}')


# Calculate and print the average accuracy across all folds
average_accuracy = sum(fold_accuracies) / len(fold_accuracies)
print(f'Average Accuracy Across Folds: {average_accuracy:.2f}') 
